# NB01 — Per-cluster Structural Coverage Extraction

**Environment:** BERDL JupyterHub (Spark)

**Purpose:** For every gene cluster with a UniProt-linkable UniRef100, assign one PDB tier (`direct` / `homolog` / `none`) and one AF tier (`confident` / `low_confidence` / `none`). Write to Parquet in MinIO.

**Inputs:**
- `kbase_ke_pangenome.bakta_annotations` — UniRef100 bridge
- `kescience_pdb.pdb_uniprot_mapping` — SIFTS chain-level mappings
- `kescience_alphafold.alphafold_entries` — AF presence per UniProt
- `kescience_alphafold.alphafold_msa_depths` — MSA depth per UniProt

**Output:**
- `s3a://cdm-lake/.../structural_coverage_biome/data/per_cluster_coverage.parquet` (~38M rows)

**Pitfalls baked in:**
- Filter out `UniRef100_UPI*` UniParc-only IDs (they don't map to AlphaFold; 22.7M rows) — validated in `alphafold_msa_annotation`.
- SIFTS is chain-level; collapse to per-UniProt best identity+coverage before tier assignment.
- Use Spark tables (not REST API) — cross-database joins only work in Spark.

## Setup

In [ ]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql.functions import regexp_replace, col, when, lit, max as spark_max

spark = get_spark_session()

OUTPUT_PATH = "s3a://cdm-lake/tenant-general-warehouse/microbialdiscoveryforge/projects/structural_coverage_biome/data/per_cluster_coverage.parquet"

## Bakta bridge (filter UPI, extract UniProt)

In [ ]:
bakta = (spark.table("kbase_ke_pangenome.bakta_annotations")
    .select("gene_cluster_id", "uniref100", "hypothetical", "ec", "kegg_orthology_id", "product")
    .filter("uniref100 IS NOT NULL AND uniref100 NOT LIKE 'UniRef100_UPI%'")
    .withColumn("uniprot_accession", regexp_replace(col("uniref100"), "UniRef100_", ""))
)
print(f"Bakta clusters with UniProt bridge: {bakta.count():,}")  # expect ~38.8M

## PDB tier — collapse SIFTS to per-UniProt best hit

In [ ]:
pdb_map = spark.table("kescience_pdb.pdb_uniprot_mapping").select(
    "uniprot_accession", "pdb_id", "identity_pct", "coverage_pct"
)
pdb_best = pdb_map.groupBy("uniprot_accession").agg(
    spark_max("identity_pct").alias("best_identity"),
    spark_max("coverage_pct").alias("best_coverage"),
)

## AF tier — presence + MSA depth

In [ ]:
af_entries = spark.table("kescience_alphafold.alphafold_entries").select(
    "uniprot_accession", "alphafold_id"
)
af_msa = spark.table("kescience_alphafold.alphafold_msa_depths").select(
    "uniprot_accession", "msa_depth"
)
af = af_entries.join(af_msa, on="uniprot_accession", how="left")

## Assign tiers

In [ ]:
per_cluster = (bakta
    .join(pdb_best, on="uniprot_accession", how="left")
    .join(af, on="uniprot_accession", how="left")
    .withColumn("pdb_tier",
        when((col("best_identity") >= 95) & (col("best_coverage") >= 80), lit("direct"))
        .when((col("best_identity") >= 30) & (col("best_identity") < 95), lit("homolog"))
        .otherwise(lit("none")))
    .withColumn("af_tier",
        when(col("alphafold_id").isNull(), lit("none"))
        .when(col("msa_depth") >= 300, lit("confident"))
        .otherwise(lit("low_confidence")))
    .select("gene_cluster_id", "uniprot_accession",
            "pdb_tier", "af_tier", "best_identity", "best_coverage", "msa_depth",
            "hypothetical", "ec", "kegg_orthology_id", "product")
)

## Sanity check tier distribution before write

In [ ]:
per_cluster.groupBy("pdb_tier", "af_tier").count().orderBy("pdb_tier", "af_tier").show()

## Write to MinIO

In [ ]:
per_cluster.write.mode("overwrite").parquet(OUTPUT_PATH)
print(f"Wrote per-cluster coverage to {OUTPUT_PATH}")